[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_12_CNN_Architecture_and_Hyperparameter_Comparison.ipynb)


# Experiment 12: CNN Architecture and Hyperparameter Comparison

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 12 – CNN Architecture and Hyperparameter Comparison

In [ ]:
# Compare compact CNN configurations on CIFAR-10 subset
import tensorflow as tf, time
(xtr0,ytr0),(xte0,yte0)=tf.keras.datasets.cifar10.load_data()
xtr0=xtr0[:10000].astype("float32")/255; ytr0=ytr0[:10000].ravel()
xv0=xte0[:2000].astype("float32")/255; yv0=yte0[:2000].ravel()
def build_cnn(nconv=2,k=3,pool="max",bn=False,lr=.001,filters=32):
    inp=tf.keras.Input((32,32,3)); z=inp
    for i in range(nconv):
        z=tf.keras.layers.Conv2D(filters*(2**min(i,2)),k,padding="same",use_bias=not bn)(z)
        if bn: z=tf.keras.layers.BatchNormalization()(z)
        z=tf.keras.layers.ReLU()(z)
        z=(tf.keras.layers.MaxPooling2D()(z) if pool=="max" else tf.keras.layers.AveragePooling2D()(z))
    z=tf.keras.layers.GlobalAveragePooling2D()(z); out=tf.keras.layers.Dense(10,activation="softmax")(z)
    m=tf.keras.Model(inp,out); m.compile(tf.keras.optimizers.Adam(lr),"sparse_categorical_crossentropy",metrics=["accuracy"]); return m
rows=[]
for nc,k,pool,bn,lr in [(2,3,"max",False,.001),(3,3,"max",True,.001),(4,3,"avg",True,.001),(2,5,"avg",False,.0005)]:
    m=build_cnn(nc,k,pool,bn,lr); t=time.time()
    h=m.fit(xtr0,ytr0,validation_data=(xv0,yv0),epochs=2,batch_size=64,verbose=0)
    rows.append([nc,k,pool,bn,lr,max(h.history["val_accuracy"]),m.count_params(),time.time()-t])
print(pd.DataFrame(rows,columns=["Conv layers","Filter size","Pooling","BatchNorm","LR","Val accuracy","Parameters","Seconds"]))